# Earthquake Detection

In [11]:
import sys
# Self-contained: the helper modules (comcat_check, plot_pick_check, plot_das_bb_check,
# detect_utilSEISMIC_fromQibin) sit next to this notebook and the small inputs are in ./data/.
# Run with the test_environment conda kernel (ELEP, pyocto, seisbench, torch, cartopy).

import gc
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seisbench.models as sbm

import torch
import pyocto
from ELEP.elep.ensemble_coherence import ensemble_semblance
from ELEP.elep.trigger_func import picks_summary_simple

from datetime import datetime
from detect_utilSEISMIC_fromQibin import apply_elep_smb, detect_on_fly, detect_on_fly2, detect_on_fly3  # local copy of DAS-denoise-detect-classify/src/models/detect_utilSEISMIC_fromQibin.py
from joblib import Parallel, delayed
from tqdm import tqdm

import obspy
from obspy import UTCDateTime, read_events, read, read_inventory
from obspy.clients.fdsn.client import Client

###
#filepath = '/fd1/QibinShi_data/akdas/qibin_data/elep_pyocto/coast_only/'
filepath = "/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/"
out_path = "/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/seismic-stations_10282026_1/association/"

## 1. Catalog Workflow for Coastal Stations Only

#### Station availability

In [12]:
# Window comes from the DAS campaign index, not a hardcoded pair of dates. These are the real
# first and last files on disk, so the broadband request cannot drift from the DAS coverage it
# will be compared against, and it re-derives itself if the campaign is ever re-indexed.
FILE_INDEX = ('/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/phasenet-3-month-run_08182026/file_index.csv')
DAS_FILE_SECONDS = 60      # each archive file holds 60 s, so the last file ENDS 60 s after it starts

_idx = pd.read_csv(FILE_INDEX, usecols=['file_time'], parse_dates=['file_time'])

starttime1 = UTCDateTime(_idx.file_time.min().isoformat())
endtime1   = UTCDateTime(_idx.file_time.max().isoformat()) + DAS_FILE_SECONDS

print('DAS index : %d files over %d days' % (len(_idx), _idx.file_time.dt.date.nunique()))
print('starttime1: %s   (first DAS file)' % starttime1)
print('endtime1  : %s   (last DAS file + %d s)' % (endtime1, DAS_FILE_SECONDS))
print('span      : %.2f days' % ((endtime1 - starttime1) / 86400))

# Coverage is not continuous inside that span -- 2023-11-09 has no data at all, and six days
# are partial. Request the whole window, but expect no DAS to associate against on those hours.
_per_day = _idx.file_time.dt.date.value_counts()
print('\npartial days (< 1440 files):')
print(_per_day[_per_day < 1440].sort_index().to_string())

DAS index : 126458 files over 90 days
starttime1: 2023-08-23T19:19:28.000000Z   (first DAS file)
endtime1  : 2023-11-22T00:00:00.000000Z   (last DAS file + 60 s)
span      : 90.19 days

partial days (< 1440 files):
file_time
2023-08-23     281
2023-09-18     471
2023-09-19     664
2023-10-14    1330
2023-11-08    1371
2023-11-10    1380


In [13]:
# Station list for the get_stations() call below, read from the FDSN text file rather than
# retyped into a string -- so adding a station to the file is the only edit needed.
STATION_TXT = 'data/MtRainier_seis.txt'   # '#Network|Station|Lat|Lon|Elev|Sitename|StartTime|EndTime'

# Keep only stations actually operating during the DAS window from the cell above. The file
# carries 1995 NEHRP temporary deployments and stations decommissioned years before this
# campaign (RVC ended 2021-01-10, WPW 2017-04-19); requesting them is silently empty, and they
# would otherwise look like stations that simply produced no picks.
FILTER_TO_DAS_WINDOW = True

_sta = pd.read_csv(STATION_TXT, sep='|')
_sta.columns = [c.lstrip('#').strip() for c in _sta.columns]
_sta['StartTime'] = pd.to_datetime(_sta['StartTime'], errors='coerce')
_sta['EndTime']   = pd.to_datetime(_sta['EndTime'],   errors='coerce')

if FILTER_TO_DAS_WINDOW:
    _t1, _t2 = pd.Timestamp(starttime1.datetime), pd.Timestamp(endtime1.datetime)
    _live = _sta[(_sta['StartTime'] <= _t2) & (_sta['EndTime'] >= _t1)]
else:
    _live = _sta

network = ','.join(sorted(_live['Network'].unique()))
station = ','.join(sorted(_live['Station'].unique()))

print('%s: %d stations, %d live in the DAS window' % (STATION_TXT, len(_sta), len(_live)))
print(_live.groupby('Network')['Station'].count().to_string())
print('\nnetwork = %r' % network)
print('station = %r' % station)

_dropped = _sta[~_sta.index.isin(_live.index)]
if len(_dropped):
    print('\ndropped (not operating in the window):')
    print(_dropped[['Network', 'Station', 'StartTime', 'EndTime']].to_string(index=False))

data/MtRainier_seis.txt: 48 stations, 33 live in the DAS window
Network
CC    25
UW     8

network = 'CC,UW'
station = 'ARAT,CARB,COPP,CRBN,FMW,GNOB,GOBB,GTWY,KAUT,LO2,LON,MILD,OBSR,OPCH,PANH,PARA,PR01,PR02,PR03,PR04,PR05,RCM,RCS,RER,RUSH,SIFT,STAR,STYX,TABR,TAVI,VOIT,WOW,WPEQ'

dropped (not operating in the window):
Network Station  StartTime             EndTime
     CC    SR41 2015-07-09 2018-04-28 00:00:00
     UW     ASF 1995-03-20 1995-06-12 23:59:59
     UW     DLR 1995-04-07 1995-09-19 23:59:59
     UW     FGM 1995-05-20 1995-09-19 23:59:59
     UW     GRL 1995-10-01 1996-07-18 23:59:59
     UW     MHL 1995-08-02 1995-10-31 23:59:59
     UW     RSH 1995-06-13 1995-09-20 23:59:59
     UW     RSU 1999-08-20 2001-09-21 00:00:00
     UW     RVC 1983-01-01 2021-01-10 00:00:00
     UW     RVN 1995-09-26 2002-08-15 00:00:00
     UW     WCR 1995-06-12 1995-09-22 23:59:59
     UW     WI2 1995-09-24 1996-07-17 23:59:59
     UW     WPS 1995-06-29 1995-09-29 23:59:59
     UW     WPW 1980-03

In [14]:
from obspy.clients.fdsn.header import URL_MAPPINGS
print(URL_MAPPINGS)

{'AUSPASS': 'http://auspass.edu.au', 'BGR': 'http://eida.bgr.de', 'EIDA': 'http://eida-federator.ethz.ch', 'ETH': 'http://eida.ethz.ch', 'EMSC': 'http://www.seismicportal.eu', 'GEONET': 'http://service.geonet.org.nz', 'GEOFON': 'http://geofon.gfz-potsdam.de', 'GFZ': 'http://geofon.gfz-potsdam.de', 'ICGC': 'http://ws.icgc.cat', 'IESDMC': 'http://batsws.earth.sinica.edu.tw', 'INGV': 'http://webservices.ingv.it', 'IPGP': 'http://ws.ipgp.fr', 'IRIS': 'http://service.iris.edu', 'IRISPH5': 'http://service.iris.edu', 'ISC': 'http://www.isc.ac.uk', 'KNMI': 'http://rdsa.knmi.nl', 'KOERI': 'http://eida.koeri.boun.edu.tr', 'LMU': 'https://erde.geophysik.uni-muenchen.de', 'NCEDC': 'https://service.ncedc.org', 'NIEP': 'http://eida-sc3.infp.ro', 'NOA': 'http://eida.gein.noa.gr', 'ODC': 'http://www.orfeus-eu.org', 'ORFEUS': 'http://www.orfeus-eu.org', 'RESIF': 'http://ws.resif.fr', 'RESIFPH5': 'http://ph5ws.resif.fr', 'RASPISHAKE': 'https://data.raspberryshake.org', 'SCEDC': 'http://service.scedc.cal

In [15]:
client = Client("IRIS")

t1 = starttime1
t2 = endtime1

inventory = client.get_stations(network=network, station = station, channel="BH?",
                                starttime=t1, endtime=t2)

inventory2 = client.get_stations(network=network, station = station, channel="HH?",
                                 starttime=t1, endtime=t2)

In [16]:
inventory

Inventory created at 2026-10-29T22:20:41.566500Z
	Created by: EarthScope WEB SERVICE: fdsnws-station | version: 1.1.59
		    https://service.earthscope.org/fdsnws/station/1/query?starttime=202...
	Sending institution: EarthScope (EarthScope)
	Contains:
		Networks (1):
			CC
		Stations (24):
			CC.ARAT (Mount Ararat)
			CC.CARB (Carbon Repeater)
			CC.COPP (Copper Mountain)
			CC.CRBN (Carbon River Ranger Station)
			CC.GNOB (Gobblers Knob Seismic)
			CC.GTWY (Gateway Entrance Station)
			CC.KAUT (Kautz Creek Helibase)
			CC.MILD (Mildred Point)
			CC.OBSR (Observation Rock)
			CC.OPCH (Ohanapecosh Administrative Area)
			CC.PANH (Panhandle Gap)
			CC.PARA (Paradise Precip Tower)
			CC.PR01 (Puyallup River 01)
			CC.PR02 (Puyallup River 02)
			CC.PR03 (Puyallup River 03)
			CC.PR04 (Puyallup River 04)
			CC.PR05 (Puyallup River 05)
			CC.RUSH (Rushingwater Creek)
			CC.SIFT (Swift Creek)
			CC.STYX (STYX, Puyallup River, WA, USA)
			CC.TABR (Tahoma Bridge)
			CC.TAVI (Tahoma Vista)
			C

In [17]:
inventory2

Inventory created at 2026-10-29T22:20:41.853700Z
	Created by: EarthScope WEB SERVICE: fdsnws-station | version: 1.1.59
		    https://service.earthscope.org/fdsnws/station/1/query?starttime=202...
	Sending institution: EarthScope (EarthScope)
	Contains:
		Networks (1):
			UW
		Stations (5):
			UW.FMW (Mt. Fremont, WA, USA)
			UW.LON (Longmire, WA, USA)
			UW.RCM (Camp Muir, Mount Rainier, WA, USA)
			UW.RER (Emerald Ridge, Mount Rainier, WA, USA)
			UW.WPEQ (White Pass Ski Facility, WA, USA)
		Channels (0):


#### Detect 30 days

In [18]:
# ### ELEP models
# devcc = "cuda"

# pn_ethz_model = sbm.EQTransformer.from_pretrained("ethz")
# pn_neic_model = sbm.EQTransformer.from_pretrained("neic")
# pn_scedc_model = sbm.EQTransformer.from_pretrained("scedc")
# pn_stead_model = sbm.EQTransformer.from_pretrained("stead")
# pn_geofon_model = sbm.EQTransformer.from_pretrained("geofon")
# pn_instance_model = sbm.EQTransformer.from_pretrained("instance")

# pn_ethz_model.to(devcc)
# pn_neic_model.to(devcc)
# pn_scedc_model.to(devcc)
# pn_stead_model.to(devcc)
# pn_geofon_model.to(devcc)
# pn_instance_model.to(devcc)

# pn_ethz_model.eval()
# pn_neic_model.eval()
# pn_scedc_model.eval()
# pn_stead_model.eval()
# pn_geofon_model.eval()
# pn_instance_model.eval()

# list_models = [pn_ethz_model,
#                pn_neic_model,
#                pn_scedc_model,
#                pn_stead_model,
#                pn_geofon_model,
#                pn_instance_model]

# ### Loop over days
# # Qibin's detect_on_fly() hardcoded endtime = t1 + 86400, so `for i in range(30)` was the day
# # chunker and the function could never see more than 24 h. This copy takes t1/t2 as arguments,
# # so the chunking has to happen HERE -- handing it the full 90-day span makes it window
# # 3.9e8 samples into ~130k windows and push them through the network in one forward call,
# # which overflows the int32 element counter in torch's pooling kernel ("integer out of range").
# # One day at 50 Hz is 1438 windows, the same size Qibin ran at Cook Inlet.
# #
# # Days are cut on UTC midnight, not on t1's 19:19:28 offset, so the '%Y%m%d' in the output
# # filename means the actual calendar day. The first and last chunks are clipped back to
# # t1 / t2 so no data outside the DAS window is requested.
# day0  = UTCDateTime(t1.date)                       # midnight of the first DAS day
# ndays = int(np.ceil((t2 - day0) / 86400))

# print('%d days: %s -> %s' % (ndays, t1, t2))

# # net_code / sta_code, NOT network / station: those two names hold the comma-separated request
# # strings built from MtRainier_seis.txt, and rebinding them here would leave them pointing at
# # whichever station this loop happened to end on -- so a later re-run of get_stations() would
# # silently request one station instead of all 33.
# for i in range(ndays):
#     d1 = max(day0 +  i      * 86400, t1)
#     d2 = min(day0 + (i + 1) * 86400, t2)

#     ### loop over stations
#     for net in inventory:
#         net_code = net.code

#         for sta in net:
#             sta_code = sta.code

#             print(net_code, sta_code, d1, d2)

#             detect_on_fly(net_code, sta_code, d1, d2, filepath, 6000, 3000, list_models, devcc)

In [19]:
### ELEP models
import os
devcc = "cuda"

pn_ethz_model = sbm.EQTransformer.from_pretrained("ethz")
pn_neic_model = sbm.EQTransformer.from_pretrained("neic")
pn_scedc_model = sbm.EQTransformer.from_pretrained("scedc")
pn_stead_model = sbm.EQTransformer.from_pretrained("stead")
pn_geofon_model = sbm.EQTransformer.from_pretrained("geofon")
pn_instance_model = sbm.EQTransformer.from_pretrained("instance")

pn_ethz_model.to(devcc)
pn_neic_model.to(devcc)
pn_scedc_model.to(devcc)
pn_stead_model.to(devcc)
pn_geofon_model.to(devcc)
pn_instance_model.to(devcc)

pn_ethz_model.eval()
pn_neic_model.eval()
pn_scedc_model.eval()
pn_stead_model.eval()
pn_geofon_model.eval()
pn_instance_model.eval()

list_models = [pn_ethz_model,
               pn_neic_model,
               pn_scedc_model,
               pn_stead_model,
               pn_geofon_model,
               pn_instance_model]

### Loop over days
# Same day chunking as the BH? cell above: detect_on_fly() takes t1/t2 rather than Qibin's
# hardcoded endtime = t1 + 86400, so the span has to be cut here. Passing the full 90 days
# gives ~130k windows in one forward call and overflows the int32 element counter in torch's
# pooling kernel ("integer out of range"). Days are cut on UTC midnight so the '%Y%m%d' in the
# output filename is a real calendar day; the first and last chunks clip back to t1 / t2.
day0  = UTCDateTime(t1.date)                       # midnight of the first DAS day
ndays = int(np.ceil((t2 - day0) / 86400))

print('%d days: %s -> %s' % (ndays, t1, t2))

# net_code / sta_code, NOT network / station: those two names hold the comma-separated request
# strings built from MtRainier_seis.txt, and rebinding them here would leave them pointing at
# whichever station this loop happened to end on -- so a later re-run of get_stations() would
# silently request one station instead of all 33.
for i in range(ndays):
    d1 = max(day0 +  i      * 86400, t1)
    d2 = min(day0 + (i + 1) * 86400, t2)

    ### loop over stations -- each inventory with ITS channel code. `inventory` was built with
    ### channel="BH?" and `inventory2` with "HH?"; no Rainier station has both, so requesting the
    ### other code is a guaranteed FDSNNoDataException ("No data for ..." for every station-day).
    ### Set CHANNEL_PASSES to just one pair to rerun a single pass.
    CHANNEL_PASSES = (("BH?", inventory), ("HH?", inventory2))
    for chan, inv in CHANNEL_PASSES:
        for net in inv:
            net_code = net.code

            for sta in net:
                sta_code = sta.code

                # Resume: detect_on_fly3 writes NET_STA_YYYYMMDD.csv, so a station-day that already
                # has one is done (the completed CC BH? pass is skipped when this cell is rerun for
                # HH?). Station-days with no data write nothing and are re-asked, which is cheap.
                if os.path.exists(os.path.join(filepath, '%s_%s_%s.csv' % (net_code, sta_code, d1.strftime('%Y%m%d')))):
                    continue

                print(net_code, sta_code, chan, d1, d2)

                detect_on_fly3(net_code, sta_code, d1, d2, filepath, 6000, 3000, list_models, devcc, channel=chan)

91 days: 2023-08-23T19:19:28.000000Z -> 2023-11-22T00:00:00.000000Z
UW FMW HH? 2023-08-23T19:19:28.000000Z 2023-08-24T00:00:00.000000Z
1683201 6000
UW LON HH? 2023-08-23T19:19:28.000000Z 2023-08-24T00:00:00.000000Z
1683201 6000
UW RCM HH? 2023-08-23T19:19:28.000000Z 2023-08-24T00:00:00.000000Z
1683201 6000
UW RER HH? 2023-08-23T19:19:28.000000Z 2023-08-24T00:00:00.000000Z
1683201 6000
UW WPEQ HH? 2023-08-23T19:19:28.000000Z 2023-08-24T00:00:00.000000Z
1683201 6000
UW FMW HH? 2023-08-24T00:00:00.000000Z 2023-08-25T00:00:00.000000Z
8640001 6000
UW LON HH? 2023-08-24T00:00:00.000000Z 2023-08-25T00:00:00.000000Z
8640001 6000
UW RCM HH? 2023-08-24T00:00:00.000000Z 2023-08-25T00:00:00.000000Z
8640001 6000
UW RER HH? 2023-08-24T00:00:00.000000Z 2023-08-25T00:00:00.000000Z
8640001 6000
UW WPEQ HH? 2023-08-24T00:00:00.000000Z 2023-08-25T00:00:00.000000Z
8640001 6000
UW FMW HH? 2023-08-25T00:00:00.000000Z 2023-08-26T00:00:00.000000Z
8640001 6000
UW LON HH? 2023-08-25T00:00:00.000000Z 2023-08-26T